In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

data = pd.read_csv("./hoteldata.csv")
data.head()

,Booking_ID,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,arrival_year,arrival_month,arrival_date,room_type_reserved.1,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
0,INN10204,NaN,NaN,NaN,2.0,Meal Plan 2,NaN,Room_Type 6,NaN,2018.0,9.0,NaN,Online,0.0,0.0,NaN,NaN,1.0,0.0
1,INN20020,NaN,NaN,NaN,2.0,Meal Plan 1,NaN,NaN,NaN,NaN,12.0,NaN,Online,0.0,0.0,0.0,NaN,NaN,0.0
2,INN16435,1.0,NaN,NaN,2.0,NaN,0.0,Room_Type 1,NaN,2018.0,11.0,NaN,NaN,0.0,0.0,NaN,NaN,1.0,0.0
3,INN07143,3.0,NaN,NaN,3.0,NaN,NaN,NaN,100.0,2018.0,5.0,NaN,Online,0.0,0.0,NaN,NaN,2.0,0.0
4,INN20511,1.0,0.0,1.0,1.0,Meal Plan 1,0.0,NaN,NaN,2018.0,11.0,NaN,NaN,0.0,0.0,0.0,150.0,NaN,1.0


In [2]:
data = data.drop(columns=['Booking_ID'])

In [3]:
x = data.drop(columns=['booking_status']) # 特征值
y = data['booking_status']                # 标签值
x.head()
# y.head()

,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,arrival_year,arrival_month,arrival_date,room_type_reserved.1,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests
0,NaN,NaN,NaN,2.0,Meal Plan 2,NaN,Room_Type 6,NaN,2018.0,9.0,NaN,Online,0.0,0.0,NaN,NaN,1.0
1,NaN,NaN,NaN,2.0,Meal Plan 1,NaN,NaN,NaN,NaN,12.0,NaN,Online,0.0,0.0,0.0,NaN,NaN
2,1.0,NaN,NaN,2.0,NaN,0.0,Room_Type 1,NaN,2018.0,11.0,NaN,NaN,0.0,0.0,NaN,NaN,1.0
3,3.0,NaN,NaN,3.0,NaN,NaN,NaN,100.0,2018.0,5.0,NaN,Online,0.0,0.0,NaN,NaN,2.0
4,1.0,0.0,1.0,1.0,Meal Plan 1,0.0,NaN,NaN,2018.0,11.0,NaN,NaN,0.0,0.0,0.0,150.0,NaN


In [4]:
x = pd.get_dummies(x,prefix=None, prefix_sep='_', dummy_na=False,columns=None, drop_first=True, dtype=None)
x = x.fillna(np.nan) # 把 DataFrame 里的所有空值填充为 np.nan


In [5]:
def handle_missing(table,columns,method='drop'):
    if columns == None:
            columns = table.columns
    for col in columns:    
        if method == 'drop':
            table[col].dropna(inplace=True)
        elif method == 'mode':  
            # table[col].fillna(table[col].mode()[0],inplace=True)   # 这个在新版本的pandas中已经没有效果，需要改为下面的写法
            table[col] = table[col].fillna(table[col].mode()[0])
        elif method == 'median':  
            table[col] = table[col].fillna(table[col].median())  
        elif method == 'mean':  
            table[col]=table[col].fillna(table[col].mean())  
        elif method == 'random':  
                table[col]=table[col].apply(lambda x: np.random.choice(table[col].dropna().values) if np.isnan(x) else x)
    return table         

In [6]:
# 查看哪些列有缺失值
missing_data_cols = x.columns[x.isnull().any()].tolist()
missing_data_cols

['no_of_adults',
 'no_of_children',
 'no_of_weekend_nights',
 'no_of_week_nights',
 'required_car_parking_space',
 'lead_time',
 'arrival_year',
 'arrival_month',
 'arrival_date',
 'repeated_guest',
 'no_of_previous_cancellations',
 'no_of_previous_bookings_not_canceled',
 'avg_price_per_room',
 'no_of_special_requests']

In [7]:
x = handle_missing(x,columns=['no_of_adults',
 'no_of_children',
 'no_of_weekend_nights',
 'no_of_week_nights',
 'required_car_parking_space',
 'lead_time',
 'arrival_year',
 'arrival_month',
 'arrival_date',
 'repeated_guest',
 'no_of_previous_cancellations',
 'no_of_previous_bookings_not_canceled',
 'avg_price_per_room',
 'no_of_special_requests'],method='mean')

x.head()

,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,required_car_parking_space,lead_time,arrival_year,arrival_month,arrival_date,repeated_guest,...,room_type_reserved_Room_Type 2,room_type_reserved_Room_Type 3,room_type_reserved_Room_Type 4,room_type_reserved_Room_Type 5,room_type_reserved_Room_Type 6,room_type_reserved_Room_Type 7,room_type_reserved.1_Complementary,room_type_reserved.1_Corporate,room_type_reserved.1_Offline,room_type_reserved.1_Online
0,1.845312,0.10351,0.807737,2.0,0.033645,86.213266,2018.000000,9.0,15.523731,0.0,...,False,False,False,False,True,False,False,False,False,True
1,1.845312,0.10351,0.807737,2.0,0.033645,86.213266,2017.820092,12.0,15.523731,0.0,...,False,False,False,False,False,False,False,False,False,True
2,1.000000,0.10351,0.807737,2.0,0.000000,86.213266,2018.000000,11.0,15.523731,0.0,...,False,False,False,False,False,False,False,False,False,False
3,3.000000,0.10351,0.807737,3.0,0.033645,100.000000,2018.000000,5.0,15.523731,0.0,...,False,False,False,False,False,False,False,False,False,True
4,1.000000,0.00000,1.000000,1.0,0.000000,86.213266,2018.000000,11.0,15.523731,0.0,...,False,False,False,False,False,False,False,False,False,False


In [8]:
x.isnull().sum()

no_of_adults                            0
no_of_children                          0
no_of_weekend_nights                    0
no_of_week_nights                       0
required_car_parking_space              0
lead_time                               0
arrival_year                            0
arrival_month                           0
arrival_date                            0
repeated_guest                          0
no_of_previous_cancellations            0
no_of_previous_bookings_not_canceled    0
avg_price_per_room                      0
no_of_special_requests                  0
type_of_meal_plan_Meal Plan 2           0
type_of_meal_plan_Meal Plan 3           0
type_of_meal_plan_Not Selected          0
room_type_reserved_Room_Type 2          0
room_type_reserved_Room_Type 3          0
room_type_reserved_Room_Type 4          0
room_type_reserved_Room_Type 5          0
room_type_reserved_Room_Type 6          0
room_type_reserved_Room_Type 7          0
room_type_reserved.1_Complementary

In [9]:
y = y.fillna(0)
y.isnull().sum()

np.int64(0)

In [10]:
# 最好做一次一步均衡处理，需要安装imbalanced-learn ： 安装好后叫做imblearn
smote = SMOTE(random_state=42)
x,y = smote.fit_resample(x,y)

In [11]:
xtrain,xtest,ytrain,ytest = train_test_split(x,y,test_size=0.2,random_state=3)

In [ ]:
#模型训练和评估
# 1.创建管线
from sklearn.model_selection import GridSearchCV


# pipe = Pipeline([('clf',xgb.XGBClassifier(use_label_encoder=False))])
pipe = Pipeline([('clf',xgb.XGBClassifier(use_label_encoder=True))])

# 2.配置参数
params={
 "clf__learning_rate"    : [0.05, 0.10, 0.15, 0.20, 0.25, 0.30 ],
 "clf__max_depth"        : [ 3, 4, 5, 6, 8, 10, 12, 15],
 "clf__min_child_weight" : [ 1, 3, 5, 7 ],
 "clf__gamma"            : [ 0.0, 0.1, 0.2 , 0.3, 0.4 ],
 "clf__colsample_bytree" : [ 0.3, 0.4, 0.5 , 0.7 ],
 "clf__subsample"        : [0.6, 0.7, 0.8, 0.9, 1.0],
 "clf__reg_alpha"        : [0, 0.001, 0.005, 0.01, 0.05],
 "clf__reg_lambda"       : [0.01, 0.1, 1.0, 10.0, 100.0]
}
# 3.创建GridSearchCV对象
cv = RandomizedSearchCV(pipe,params,cv=5,scoring='accuracy')
# 4.训练
cv.fit(xtrain,ytrain)
# 5.模型预测
ypred = cv.predict(xtest)
# 6.模型评估
print("Accuracy: {}".format(cv.score(xtest, ytest)))
print("Tuned Model Parameters: {}".format(cv.best_params_))

d:\programs\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:18:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\programs\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:18:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\programs\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:18:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\programs\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:18:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iterati

Accuracy: 0.8260083606372162
Tuned Model Parameters: {'clf__subsample': 1.0, 'clf__reg_lambda': 1.0, 'clf__reg_alpha': 0.001, 'clf__min_child_weight': 1, 'clf__max_depth': 15, 'clf__learning_rate': 0.05, 'clf__gamma': 0.2, 'clf__colsample_bytree': 0.3}
